In [9]:
import pandas as pd
import numpy as np
import pickle

from ast import literal_eval
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 1. LOAD DATA
# ============================================================

movies = pd.read_csv(r"D:\downloads\tmdb_5000_movies.csv")
credits = pd.read_csv(r"D:\downloads\tmdb_5000_credits.csv")

print("Movies shape:", movies.shape)
print("Credits shape:", credits.shape)


# ============================================================
# 2. SELECT REQUIRED COLUMNS
# ============================================================

movies = movies[
    [
        "id",
        "title",
        "overview",
        "genres",
        "keywords"
    ]
]

credits = credits[
    [
        "movie_id",
        "cast",
        "crew"
    ]
]


# ============================================================
# 3. RENAME COLUMN
# ============================================================

movies.rename(
    columns={"id": "movie_id"},
    inplace=True
)


# ============================================================
# 4. MERGE DATASETS
# ============================================================

movies = movies.merge(
    credits,
    on="movie_id"
)

print("\nAfter merging:")
print(movies.shape)


# ============================================================
# 5. CHECK MISSING VALUES
# ============================================================

print("\nMissing values:")
print(movies.isnull().sum())


# ============================================================
# 6. REMOVE ROWS WITH MISSING VALUES
# ============================================================

movies.dropna(inplace=True)


# ============================================================
# 7. CONVERT JSON-LIKE DATA INTO LISTS
# ============================================================

def convert_to_list(text):

    result = []

    for item in literal_eval(text):

        result.append(item["name"])

    return result


movies["genres"] = movies["genres"].apply(
    convert_to_list
)

movies["keywords"] = movies["keywords"].apply(
    convert_to_list
)

movies["cast"] = movies["cast"].apply(
    convert_to_list
)


# ============================================================
# 8. GET DIRECTOR FROM CREW
# ============================================================

def get_director(text):

    result = []

    for item in literal_eval(text):

        if item["job"] == "Director":

            result.append(item["name"])

    return result


movies["crew"] = movies["crew"].apply(
    get_director
)


# ============================================================
# 9. CONVERT OVERVIEW INTO LIST
# ============================================================

movies["overview"] = movies["overview"].apply(
    lambda x: x.split()
)


# ============================================================
# 10. REMOVE SPACES FROM WORDS
# ============================================================

movies["genres"] = movies["genres"].apply(
    lambda x: [i.replace(" ", "") for i in x]
)

movies["keywords"] = movies["keywords"].apply(
    lambda x: [i.replace(" ", "") for i in x]
)

movies["cast"] = movies["cast"].apply(
    lambda x: [i.replace(" ", "") for i in x]
)

movies["crew"] = movies["crew"].apply(
    lambda x: [i.replace(" ", "") for i in x]
)


# ============================================================
# 11. CREATE COMBINED TAGS
# ============================================================
#instead of treating all these columns separately, we create one combined representation.called as tag
movies["tags"] = (
    movies["overview"]
    + movies["genres"]
    + movies["keywords"]
    + movies["cast"]
    + movies["crew"]
)


# ============================================================
# 12. CONVERT TAGS INTO TEXT
# ============================================================

movies["tags"] = movies["tags"].apply(
    lambda x: " ".join(x)
)


# ============================================================
# 13. CONVERT TO LOWERCASE
# ============================================================

movies["tags"] = movies["tags"].apply(
    lambda x: x.lower()
)


# ============================================================
# 14. CREATE FINAL DATAFRAME
# ============================================================

new_movies = movies[
    [
        "movie_id",
        "title",
        "tags"
    ]
].copy()


print("\nFinal dataframe:")
print(new_movies.head())


# ============================================================
# 15. TEXT VECTORIZATION
# ============================================================
#It converts every movie's tags into numbers
cv = CountVectorizer(
    max_features=5000,
    stop_words="english"
)

vectors = cv.fit_transform(
    new_movies["tags"]
).toarray()

print("\nVector shape:")
print(vectors.shape)


# ============================================================
# 16. COSINE SIMILARITY
# ============================================================

similarity = cosine_similarity(vectors)

print("\nSimilarity matrix shape:")
print(similarity.shape)


# ============================================================
# 17. RECOMMENDATION FUNCTION
# ============================================================

def recommend(movie_name):

    movie_index = new_movies[
        new_movies["title"] == movie_name
    ].index[0]

    distances = similarity[movie_index]

    movie_indices = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    print("\nMovies recommended for:", movie_name)

    for index, score in movie_indices:

        movie = new_movies.iloc[index]["title"]

        print(
            movie,
            " | Similarity:",
            round(score, 3)
        )


# ============================================================
# 18. TEST RECOMMENDATION
# ============================================================

recommend("Avatar")


# ============================================================
# 19. SAVE MOVIE LIST
# ============================================================

pickle.dump(
    new_movies,
    open("model/movie_list.pkl", "wb")
)


# ============================================================
# 20. SAVE SIMILARITY MATRIX
# ============================================================

pickle.dump(
    similarity,
    open("model/similarity.pkl", "wb")
)


print("\n===================================")
print("Movie recommender model saved!")
print("===================================")

Movies shape: (4803, 20)
Credits shape: (4803, 4)

After merging:
(4803, 7)

Missing values:
movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

Final dataframe:
   movie_id                                     title  \
0     19995                                    Avatar   
1       285  Pirates of the Caribbean: At World's End   
2    206647                                   Spectre   
3     49026                     The Dark Knight Rises   
4     49529                               John Carter   

                                                tags  
0  in the 22nd century, a paraplegic marine is di...  
1  captain barbossa, long believed to be dead, ha...  
2  a cryptic message from bond’s past sends him o...  
3  following the death of district attorney harve...  
4  john carter is a war-weary, former military ca...  

Vector shape:
(4800, 5000)

Similarity matrix shape:
(4800, 4800)

Movies recommended for: Avatar
Lifefo

In [3]:
import os

print("Notebook is running from:")
print(os.getcwd())

Notebook is running from:
C:\Users\mdnom


In [4]:
import os

os.makedirs("model", exist_ok=True)

print("model folder created!")

model folder created!


In [5]:
import os
import pickle

# Create the model folder inside the PyCharm project
model_path = r"C:\Users\mdnom\PycharmProjects\movie-recmmend\model"

os.makedirs(model_path, exist_ok=True)

# Save the two files directly there
pickle.dump(
    new_movies,
    open(os.path.join(model_path, "movie_list.pkl"), "wb")
)

pickle.dump(
    similarity,
    open(os.path.join(model_path, "similarity.pkl"), "wb")
)

print("Both files saved successfully!")
print(model_path)

Both files saved successfully!
C:\Users\mdnom\PycharmProjects\movie-recmmend\model


In [6]:
movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver, ...",[JamesCameron],"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley, Ste...",[GoreVerbinski],"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux, Ralp...",[SamMendes],a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman, Anne...",[ChristopherNolan],following the death of district attorney harve...
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton, Wi...",[AndrewStanton],"john carter is a war-weary, former military ca..."


In [8]:
print(movies['tags'][0])

in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver stephenlang michellerodriguez giovanniribisi joeldavidmoore cchpounder wesstudi lazalonso dileeprao mattgerald seananthonymoran jasonwhyte scottlawrence kellykilgour jamespatrickpitt seanpatrickmurphy peterdillon kevindorman kelsonhenderson davidvanhorn jacobtomuri michaelblain-rozgay joncurry lukehawker woodyschultz petermensah soniayee jahnelcurfman ilramchoi kylawarren lisaroumain debrawilson chrismala taylorkibby jodielandau julielamm cullenb.madden josephbradymadden frankietorres austinwilson sarawilson tamicawashington-miller lucybriant nathanme

In [ ]:
#The notebook is used to make/build the recommendation system.
#The app.py is used to give that recommendation system to the user as an application.
#Without app.py, you would have to open your notebook and write:
#recommend("Avatar")
#every time.
#That's not convenient for a normal user.
#app.py
#→ Only needed if you want to turn the model into an application/interface that a user can interact with.